In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Notebook runs from ai_agentic/gen-ai-work
env_path = Path.cwd().parent / ".env"

print("Looking for .env at:", env_path)

if not env_path.exists():
    raise FileNotFoundError(f".env not found at {env_path}")

load_dotenv(env_path, override=True)

load_dotenv(dotenv_path=env_path, override=True)

print("Loading env from:", env_path)

print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))


from typing import Any, Dict

from google.adk.agents import Agent, LlmAgent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.google_llm import Gemini
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ ADK components imported successfully.")

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)


# Helper functions
#await run_session(runner, "What is the capital of France?", "geography-session")
#await run_session(runner, ["Hello!", "What's my name?"], "user-intro-session")


# Define helper functions that will be reused throughout the notebook
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n ### Session: {session_name}")

    # Get app name from the Runner
    app_name = runner_instance.app_name

    # Attempt to create a new session or retrieve an existing one
    try:
        session = await session_service.create_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )
    except:
        session = await session_service.get_session(
            app_name=app_name, user_id=USER_ID, session_id=session_name
        )

    # Process queries if provided
    if user_queries:
        # Convert single query to list for uniform processing
        if type(user_queries) == str:
            user_queries = [user_queries]

        # Process each query in the list sequentially
        for query in user_queries:
            print(f"\nUser > {query}")

            # Convert the query string to the ADK Content format
            query = types.Content(role="user", parts=[types.Part(text=query)])

            # Stream the agent's response asynchronously
            async for event in runner_instance.run_async(
                user_id=USER_ID, session_id=session.id, new_message=query
            ):
                # Check if the event contains valid content
                if event.content and event.content.parts:
                    # Filter out empty or "None" responses before printing
                    if (
                        event.content.parts[0].text != "None"
                        and event.content.parts[0].text
                    ):
                        print(f"{MODEL_NAME} > ", event.content.parts[0].text)
    else:
        print("No queries!")


print("✅ Helper functions defined.")




Looking for .env at: /Users/anujmittal/Desktop/ai_agentic/.env
Loading env from: /Users/anujmittal/Desktop/ai_agentic/.env
GOOGLE_API_KEY loaded: True
✅ ADK components imported successfully.
✅ Helper functions defined.


In [ ]:
# Simple stateful agent with in memory session
APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session

MODEL_NAME = "gemini-2.5-flash-lite"


# Step 1: Create the LLM Agent
root_agent = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot",  # Description of the agent's purpose
)

# Step 2: Set up Session Management
# InMemorySessionService stores conversations in RAM (temporary)
session_service = InMemorySessionService()

# Step 3: Create the Runner
runner = Runner(
    agent=root_agent, 
    app_name=APP_NAME, 
    session_service=session_service
    )

print("Stateful agent initialized!")
print(f"   - Application: {APP_NAME}")
print(f"   - User: {USER_ID}")
print(f"   - Using: {session_service.__class__.__name__}")

# Testing stateful agent
await run_session(
    runner,
    [
        "Hi, I am Sam! What is the capital of United States?",
        "Hello! What is my name?",  # This time, the agent should remember!
    ],
    "stateful-agentic-session",
)

# Run this cell after restarting the kernel. All this history will be gone...
await run_session(
    runner,
    ["What did I ask you about earlier?", "And remind me, what's my name?"],
    "stateful-agentic-session",
)  

Stateful agent initialized!
   - Application: default
   - User: default
   - Using: InMemorySessionService

 ### Session: stateful-agentic-session

User > Hi, I am Sam! What is the capital of United States?


In [ ]:
# Implementing Persistent Sessions

# Step 1: Create the same agent (notice we use LlmAgent this time)
chatbot_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="text_chat_bot",
    description="A text chatbot with persistent memory",
)

#Step 2: Create database session service

from pathlib import Path
PROJECT_ROOT = Path.cwd().parent   # because notebook is in gen-ai-work/
DB_PATH = PROJECT_ROOT / "data" / "agent_sessions.db"

DATABASE_URL = f"sqlite:///{DB_PATH}"

print("Using SQLite DB:", DATABASE_URL)

#db_url="sqlite:///my_agent_data.db"
session_service=DatabaseSessionService(db_url=DATABASE_URL)

#Step 3: Create runner with db storage
runner=Runner(
    agent=chatbot_agent,
    session_service=session_service,
    app_name=APP_NAME
)

print("runner created with persistent service.")

#Test it
await run_session(
    runner,
    ["Hi, I am Sam! What is the capital of the United States?", "Hello! What is my name?"],
    "test-db-session-01",
)


#restart the kernel and comment the above step to test forgotness of agent
await run_session(
    runner,
    ["Hi, I am Sam! What is the capital of the United States?", "Hello! What is my name?"],
    "test-db-session-01",
)